<NumPy로 Attention 구현하기>

- NumPy만 사용하여 기본적인 attention 함수를 직접 구현하시오.
- 요구 사항
    - softmax(x) 함수를 직접 구현할 것
    - attention(query, keys, values) 함수를 구현할 것
    - attention score, attention weight, context vector를 출력할 것
    - 어떤 key와 value가 더 크게 반영되었는지 간단히 설명할 것

In [14]:
import numpy as np

##1. softmax(x)
def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis = 0)

##2. attention(query, keys, values)
def attention(query, keys, values):
    ## attention scores
    scores = np.dot(query, keys.T)
    ## softmax
    weights = softmax(scores)
    ## context vector
    context = np.dot(weights, values)
    return scores, weights, context

##3. 테스트
query = np.array([1,2,3])

keys = np.array([
    [1,2,3],
    [3,1,2],
    [0,4,5]
])

values = np.array([
    [100, 0],
    [0, 100],
    [0, 0]
])

scores, weights, context = attention(query, keys, values)
print("Attention Scores:", scores)
print("Attention Weights:", weights)
print("Context Vector:", context)


Attention Scores: [14 11 23]
Attention Weights: [1.23393818e-04 6.14341645e-06 9.99870463e-01]
Context Vector: [0.01233938 0.00061434]


Q. query와 관련 아예 없는 [0,4,5]로 테스트 해봤을때, 내적때문에 가중치가 key3에 쏠리게 되는 것 같은데, 잘못되었다는 것을 어떻게 아나요?

A. 그래서 scaled dot product attention을 통해 크기만큼 보존을 해줘야 합니다.

#### Scaled Dot-Product Attention
- 기존의 내적(Dot-Product) attention 결과값이 모델의 차원이 커질수록 수학적으로 폭발하는 현상을 방지하기 위해, 내적 값을 벡터 차원 수의 제곱근($\sqrt{d_k}$)으로 나누어(Scale) 안정화시킨 트랜스포머(Transformer) 아키텍처의 핵심 연산 기법 
$$
Attention(Q,K,V) = softmax( {\frac{Q*K^T}{\sqrt {d_k}}}) * V
$$

In [17]:
import numpy as np

##1. softmax(x)
def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis = 0)

##2. attention(query, keys, values)
def scaled_dot_product_attention(query, keys, values):
    d_k = keys.shape[-1]
    ## attention scores
    scores = np.dot(query, keys.T) / np.sqrt(d_k)
    ## softmax
    weights = softmax(scores)
    ## context vector
    context = np.dot(weights, values)
    return scores, weights, context

##3. 테스트
query = np.array([1,2,3])

keys = np.array([
    [1,2,3],
    [3,1,2],
    [0,4,5]
])

values = np.array([
    [100, 0],
    [0, 100],
    [0, 0]
])

scores, weights, context = scaled_dot_product_attention(query, keys, values)
print("Attention Scores:", scores)
print("Attention Weights:", weights)
print("Context Vector:", context)


Attention Scores: [ 8.08290377  6.35085296 13.27905619]
Attention Weights: [5.50197112e-03 9.73415368e-04 9.93524614e-01]
Context Vector: [0.55019711 0.09734154]
